In [17]:
# ── Mermaid rendering helper (run this cell once) ──
import base64
from IPython.display import Image, display

def mermaid(code, width=800):
    """Render a Mermaid diagram as an image via mermaid.ink."""
    encoded = base64.urlsafe_b64encode(code.encode('utf-8')).decode('ascii')
    url = f'https://mermaid.ink/img/{encoded}'
    display(Image(url=url, width=width))

### Lesson 2.1 - Vanilla RNNs and Backpropagation Through Time (BPTT)
---
> *Note:* This lesson and all the forthcoming lessons will have some sort of images/mermaid diagrams to help the reader.

<b>2.1.1: The Problem with Feedforward Networks</b><br>

In the earlier modules we worked with BoW, TF-IDF and Word2Vec. But all of them draw a big limitation:

**Problem:** These models ignore the sequence/ordering of the words in a sentence

```bash
Sentence 1: "I love cats"
Sentence 2: "Cats love I"

BoW/TF-IDF: Vector for both the sentences will be same but the meaning is totally different!
```

**Question:** How can we preserve the sequence of the words.<br>
**Answer: Recurrent Neural Network(RNN)** - Process a sequence!

---
<b>2.1.2: Core concept of RNN - The memory</b><br>

**Intuition:**<br>
When you read a sentence, you brain tends to remember the previous words as you go on. RNN does the same!

**RNN has a "Hidden State" (h)** - this is its memory that remembers the information of previous words in the sequence.

**Key Insight:**

- At every time step (t), RNN keeps track of 2 things:
    1. Current input ($x_t$) - the current word in the sequence
    2. Previous hidden state ($h_{t-1}$) - the memory of previous words
- New memory is obtained by combining both ($h_t$)

---
<b>2.1.3: RNN Architecture - Unrolled View</b><br>

"unroll"-ing an RNN means - breaking it in time steps.

In [18]:
mermaid("""
graph TB
    subgraph "RNN Cell"
        direction LR
        xt["x_t<br/>Current Input"] --> concat["Concatenate"]
        ht_1["h_{t-1}<br/>Previous Memory"] --> concat
        concat --> tanh["tanh Activation"]
        tanh --> ht["h_t<br/>New Memory"]
        ht --> yt["y_t<br/>Output"]
    end

    style xt fill:#ff9999,stroke:#333,stroke-width:2px
    style ht_1 fill:#99ccff,stroke:#333,stroke-width:2px
    style ht fill:#99ff99,stroke:#333,stroke-width:2px
    style concat fill:#ffff99
    style tanh fill:#ff99ff
""")

**Components:**

1. $x_t$: Input at current time step.
2. $h_{t-1}$: Memory of previous time step.
3. Concatenate: Combining both above
4. tanh (tansh): Activation function (compress the values between -1 and 1)
5. $h_{t}$: New memory (that will be passed in the new step)
6. $y_t$: Output

---
<b>2.1.4: RNN Through Time - Complete Sequence</b><br>

At the time of processing a whole sentence:

In [19]:
mermaid("""
sequenceDiagram
    participant Input as Input Sequence
    participant RNN1 as RNN t=1
    participant RNN2 as RNN t=2
    participant RNN3 as RNN t=3
    participant RNN4 as RNN t=4
    participant Output as Hidden States
    
    Input->>RNN1: x₁ = "I"
    RNN1->>RNN1: h₀ = [0,0,0,...] (initial)
    RNN1->>Output: h₁ = memory after "I"
    
    Input->>RNN2: x₂ = "love"
    RNN2->>RNN2: h₁ (from previous)
    RNN2->>Output: h₂ = memory after "love"
    
    Input->>RNN3: x₃ = "cats"
    RNN3->>RNN3: h₂ (from previous)
    RNN3->>Output: h₃ = memory after "cats"
    
    Input->>RNN4: x₄ = "!"
    RNN4->>RNN4: h₃ (from previous)
    RNN4->>Output: h₄ = final memory
""")

Notice:

- At every step the previous memory is being passed
- Final hidden state (h_4) has information about the whole sentence
- This memory can be used for any task (Classification, translation, etc)

---
<b>2.1.5: Backpropagation Through Time (BPTT)</b><br>

At the time of training, we need to calculate the gradients. RNN has a special twist:

**Problem:** Loss is calculated at the final time step, but gradient has to go to all the previous time steps

In [20]:
mermaid("""
graph BT
    subgraph "Forward Pass (Left to Right)"
        h1_f["h₁"] --> h2_f["h₂"]
        h2_f --> h3_f["h₃"]
        h3_f --> h4_f["h₄"]
        h4_f --> Loss["Loss Calculation"]
    end
    
    subgraph "Backward Pass (Right to Left - BPTT)"
        Loss -.->|∂Loss/∂h₄| h4_b["h₄"]
        h4_b -.->|∂Loss/∂h₃| h3_b["h₃"]
        h3_b -.->|∂Loss/∂h₂| h2_b["h₂"]
        h2_b -.->|∂Loss/∂h₁| h1_b["h₁"]
    end
    
    style Loss fill:#ff6666,stroke:#333,stroke-width:3px
    style h1_f fill:#99ccff
    style h2_f fill:#99ccff
    style h3_f fill:#99ccff
    style h4_f fill:#99ccff
    style h1_b fill:#ffcc99
    style h2_b fill:#ffcc99
    style h3_b fill:#ffcc99
    style h4_b fill:#ffcc99
""")

**Key Insights:**

- Gradients needs to be passed through all the time steps
- Thats why it is named Backpropagation Through Time (BPTT)
- This is computationally heavy (specially for long sequences)

---

### Math behind it:
---

<b>2.1.6: The RNN Equation</b><br>

Forward pass:

$h_t = tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$

Breakdown:

- $x_t$: Current input
- $h_{t-1}$: Previous hidden state
- $W_{xh}$: Input to hidden weights
- $W_{hh}$: Hidden to hidden weights
- $b_h$: Bias
- tanh: Activation function

Matrix multiplication flow:

```bash
x_t @ W_xh → (batch, input) @ (input, hidden) → (batch, hidden)
h_{t-1} @ W_hh → (batch, hidden) @ (hidden, hidden) → (batch, hidden)
Add both + bias → (batch, hidden)
Apply tanh → (batch, hidden) = h_t
```

In [24]:
# VanillaRNNCell
import torch
import torch.nn as nn

class VanillaRNNCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()

        self.W_xh = nn.Parameter(torch.randn(input_dim, hidden_dim))
        self.W_hh = nn.Parameter(torch.randn(hidden_dim, hidden_dim))
        self.b_h = nn.Parameter(torch.randn(hidden_dim))

    def forward(self, x_t, h_prev):
        h_t = torch.tanh(x_t @ self.W_xh + h_prev @ self.W_hh + self.b_h)
        return h_t

input_dim = 10
hidden_dim = 20
batch_size = 4

cell = VanillaRNNCell(input_dim, hidden_dim)
x_t = torch.randn(batch_size, input_dim)
h_prev = torch.randn(batch_size, hidden_dim)

h_t = cell(x_t, h_prev)
print(f"Input shape: {x_t.shape}")
print(f"Previous hidden shape: {h_prev.shape}")
print(f"New hidden shape: {h_t.shape}")

Input shape: torch.Size([4, 10])
Previous hidden shape: torch.Size([4, 20])
New hidden shape: torch.Size([4, 20])


In [26]:
# VanillaRNN

class VanillaRNN(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()

        self.cell = VanillaRNNCell(input_dim, hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x_sequence):
        batch_size, seq_len, input_dim = x_sequence.shape

        h_prev = torch.zeros(batch_size, self.hidden_dim)

        hidden_states = []

        for t in range(seq_len):
            x_t = x_sequence[:, t, :]
            h_t = self.cell(x_t, h_prev)
            hidden_states.append(h_t)
            h_prev = h_t

        output = torch.stack(hidden_states, dim=1)
        return output


# test it
rnn = VanillaRNN(input_dim=10, hidden_dim=20)
x_seq = torch.randn(4, 5, 10)

output = rnn(x_seq)
print(f"Input shape: {x_seq.shape}")
print(f"Output shape: {output.shape}")

Input shape: torch.Size([4, 5, 10])
Output shape: torch.Size([4, 5, 20])


In [28]:
import torch.optim as optim

rnn = VanillaRNN(input_dim=10, hidden_dim=20)
criterion = nn.MSELoss()
optimizer = optim.Adam(rnn.parameters(), lr=0.01)

epochs = 100

for epoch in range(epochs):
    optimizer.zero_grad()
    output = rnn(x_seq)
    target = torch.randn(4, 5, 20)
    loss = criterion(output, target)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch + 1} | Loss: {loss.item():.4f}")

Epoch: 10 | Loss: 2.0964
Epoch: 20 | Loss: 1.7288
Epoch: 30 | Loss: 1.9094
Epoch: 40 | Loss: 1.8542
Epoch: 50 | Loss: 1.7964
Epoch: 60 | Loss: 1.8435
Epoch: 70 | Loss: 1.6572
Epoch: 80 | Loss: 1.8268
Epoch: 90 | Loss: 1.8653
Epoch: 100 | Loss: 1.8187
